# Project 01 — E-commerce Data Cleaning

**Author:** Yeganeh Malakuti

This notebook loads the raw e-commerce dataset, identifies data-quality
issues (missing values, duplicates, invalid values, wrong dtypes), fixes
them, and exports a cleaned dataset for downstream analysis.


In [ ]:
import pandas as pd
import numpy as np


## 1. Load and inspect the dataset


In [ ]:
# Load the raw dataset. Path is relative to this notebook's location,
# so the notebook works on any machine without editing the path.
RAW_DATA_PATH = "../dataset/First Dataset.xlsx"

df = pd.read_excel(RAW_DATA_PATH)


In [ ]:
# Quick structural overview: column types, non-null counts, and
# summary statistics for numeric columns.
print(df.head())
print(df.info())
print(df.describe())


## 2. Remove duplicate records


In [ ]:
# Inspect duplicate rows before removing them, ignoring customer_id
# since it is expected to be unique per row and would hide true
# duplicates in the other columns.
df[df.duplicated(subset=df.columns.drop("customer_id"))]


In [ ]:
# Drop exact duplicate records (all columns except customer_id),
# keeping the first occurrence.
df = df.drop_duplicates(
    subset=df.columns.drop("customer_id"),
    keep="first"
)


There are no duplicated rows remaining.


## 3. Clean `first_name` and `gender`


In [ ]:
# List unique first names to check for inconsistent gender labels.
print(sorted(df["first_name"].unique()))


In [ ]:
# Reference mapping of first name -> expected gender, built from
# manual inspection of the unique names above.
name_gender = {
    "Ali": "M",
    "Amir": "M",
    "Arash": "M",
    "Parsa": "M",
    "Reza": "M",
    "Sina": "M",
    "Kimia": "F",
    "Maryam": "F",
    "Mina": "F",
    "Neda": "F",
    "Sara": "F",
    "Zahra": "F",
}

# Overwrite gender wherever it doesn't match the expected value for
# that first name.
for name, gender in name_gender.items():
    mask = (df["first_name"] == name) & (df["gender"] != gender)
    df.loc[mask, "gender"] = gender


In [ ]:
# Verify no mismatches remain.
for name, gender in name_gender.items():
    mismatch = df[(df["first_name"] == name) & (df["gender"] != gender)]
    if not mismatch.empty:
        print(mismatch)


After correction, every name matches its expected gender.


## 4. Clean `age`


In [ ]:
# Fill missing ages with the median age (robust to outliers).
median_age = df["age"].median()
print(f"Median age used for imputation: {median_age}")
df["age"] = df["age"].fillna(median_age)


In [ ]:
# Cap unrealistic ages (> 110) by replacing them with a reasonable
# fallback value.
df.loc[df["age"] > 110, "age"] = 45


In [ ]:
# Age is a whole number of years, so store it as an integer.
df["age"] = df["age"].astype(np.int64)


No missing values or outliers remain in `age`.


## 5. Recalculate `total_spending`


In [ ]:
# total_spending should equal purchase_count * avg_order_value.
# Compare the stored value against the expected value to find errors.
expected_total = df["purchase_count"] * df["avg_order_value"]

comparison = pd.DataFrame({
    "Purchase Count": df["purchase_count"],
    "Average Order Value": df["avg_order_value"],
    "Current Total Spending": df["total_spending"],
    "Expected Total Spending": expected_total,
})

incorrect_spending = comparison[
    comparison["Current Total Spending"] != comparison["Expected Total Spending"]
]
print(incorrect_spending)

# Recompute total_spending directly from its components so every row
# is internally consistent.
df["total_spending"] = df["purchase_count"] * df["avg_order_value"]


All incorrect `total_spending` values are now corrected.


## 6. Fix invalid `returned_items`


In [ ]:
# returned_items can never exceed purchase_count. Flag rows where
# this constraint is violated.
invalid_returns = df[df["returned_items"] > df["purchase_count"]]
print(invalid_returns)


In [ ]:
# There is no reliable way to infer the true value for these rows,
# so they are set to missing rather than guessed.
mask = df["returned_items"] > df["purchase_count"]
df.loc[mask, "returned_items"] = pd.NA


There is no way to determine the real returned-items value for
these rows without risking incorrect assumptions, so they are left
as missing.


## 7. Fix data types


In [ ]:
# Low-cardinality text columns are stored as category dtype for
# memory efficiency and to signal that they hold a fixed set of
# labels.
categorical_columns = [
    "gender",
    "city",
    "province",
    "membership_tier",
    "payment_method",
    "device",
    "discount_used",
]

df[categorical_columns] = df[categorical_columns].astype("category")


In [ ]:
# Use pandas' nullable string dtype for free-text names.
df["first_name"] = df["first_name"].astype("string")


In [ ]:
# Parse signup_date as a proper datetime instead of a string.
df["signup_date"] = pd.to_datetime(df["signup_date"])


In [ ]:
# Nullable integer dtype, since returned_items can be missing (NA)
# but is otherwise a whole number.
df["returned_items"] = df["returned_items"].astype("Int64")


In [ ]:
# Final structural check to confirm all dtypes were applied.
df.info()


## 8. Export the cleaned dataset


In [ ]:
# Save the cleaned dataset into this submission folder.
OUTPUT_PATH = "cleaned_dataset_yeganeh-malakuti.xlsx"
df.to_excel(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to {OUTPUT_PATH}")
